Model Selection & Regularization


In [2]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.metrics import classification_report

In [3]:
df = pd.read_csv('diabetes_012_health_indicators_BRFSS2015.csv')
df.head()


,Diabetes_012,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0


# Define the Prediction and feature variables
Categorical outcome variable = diabetes_status.  
Outcome: whether the patient is diabetic or non-diabetic (The original data set has three categories, but for a better analysis and as in medical fields diabetic and prediabetic are treated the same, we are going to keep two categories only)  

1 = diabetic  
0 = non-diabetic  


In [4]:
df['diabetes_status'] = df['Diabetes_012'].replace({0:0, 1:1, 2:1})
target = 'diabetes_status'


1. BMI :  A higher BMI is a stronger risk factor for diabetes.  
2. HighBP : Hypertension is highly associated with diabetes.
3. Age : Diabetes prevalence increases significantly with age.
4. GenHlth : general health is strongly associated with chronic diseases as diabetes.
5. PhysActivity : Lack of physical activity may increase diabetes risk.
6. HighChol  : Diabetes and high cholesterol are frequently associated.
7. DiffWalk  : Difficulty walking often reflects obesity and diabetic complications.
8. HeartDiseaseorAttack : Cardiovascular disease is strongly associated with diabetes.
9. Education : People with less education may have more difficulty recognizing the diabetes signs.

In [5]:
features = [
    'BMI',
    'HighBP',
    'Age',
    'Education',
    'GenHlth',
    'PhysHlth',
    'HighChol',
    'DiffWalk',
    'HeartDiseaseorAttack'
]

In [6]:
df['diabetes_status'].value_counts()

diabetes_status
0.0    213703
1.0     39977
Name: count, dtype: int64

# Data Preparation

In [7]:
df.isnull().sum() #check for missing values

Diabetes_012            0
HighBP                  0
HighChol                0
CholCheck               0
BMI                     0
Smoker                  0
Stroke                  0
HeartDiseaseorAttack    0
PhysActivity            0
Fruits                  0
Veggies                 0
HvyAlcoholConsump       0
AnyHealthcare           0
NoDocbcCost             0
GenHlth                 0
MentHlth                0
PhysHlth                0
DiffWalk                0
Sex                     0
Age                     0
Education               0
Income                  0
diabetes_status         0
dtype: int64

In [8]:
# Separate data into X(features) and y (outcome)
X = df[features]
y = df[target]

In [9]:
# One-hot encoding Education variable

X = pd.get_dummies(
    X,
    columns=['Education'],
    drop_first=True
)

In [10]:
from sklearn.model_selection import train_test_split

# Split data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Check obserbation counts in each class
y_train.value_counts()

diabetes_status
0.0    170946
1.0     31998
Name: count, dtype: int64

The data set is highly imbalanced. Most obserbations belong to the "No diabetes" class. The diabetes class (pre-diabetic and diabetic) has far fewer observations. This imbalance can cause our classification models to favor the majority class and perform poorly when predicting the minority class. So, balancing the training dataset would be the right approach before modeling.

In [11]:
train_df = X_train.copy()
train_df['diabetes_status'] = y_train


In [12]:
# Separate majority and minority class
majority_class = train_df[train_df['diabetes_status'] == 0]
minority_class = train_df[train_df['diabetes_status'] == 1]

# Undersample the majority class by randomly sampling it
majority_class_undersampled = majority_class.sample(n=len(minority_class), random_state=42)

# Combine the undersampled majority class with the minority class
train_balanced = pd.concat([majority_class_undersampled, minority_class])

# Shuffle the resulting dataframe to ensure randomness
train_balanced = train_balanced.sample(frac=1, random_state=42).reset_index(drop=True)
train_balanced

,BMI,HighBP,Age,GenHlth,PhysHlth,HighChol,DiffWalk,HeartDiseaseorAttack,Education_2.0,Education_3.0,Education_4.0,Education_5.0,Education_6.0,diabetes_status
0,21.0,0.0,7.0,1.0,0.0,0.0,0.0,0.0,False,False,True,False,False,0.0
1,24.0,0.0,4.0,4.0,0.0,0.0,0.0,0.0,False,False,False,True,False,0.0
2,25.0,0.0,3.0,3.0,0.0,0.0,0.0,0.0,False,False,False,False,True,0.0
3,27.0,0.0,9.0,2.0,0.0,1.0,0.0,0.0,False,False,False,False,True,0.0
4,46.0,1.0,7.0,4.0,20.0,1.0,1.0,0.0,False,False,True,False,False,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63991,27.0,0.0,12.0,4.0,2.0,1.0,0.0,1.0,False,False,True,False,False,1.0
63992,30.0,0.0,6.0,3.0,14.0,0.0,0.0,0.0,False,False,False,False,True,1.0
63993,26.0,1.0,10.0,2.0,5.0,1.0,0.0,0.0,False,False,False,False,True,0.0
63994,22.0,0.0,10.0,5.0,30.0,0.0,0.0,0.0,False,False,False,True,False,0.0


In [13]:
train_balanced['diabetes_status'].value_counts()

diabetes_status
0.0    31998
1.0    31998
Name: count, dtype: int64

# Model Selection with 5-fold Cross Validation
Using only training data:


In [14]:
train_balanced.columns

Index(['BMI', 'HighBP', 'Age', 'GenHlth', 'PhysHlth', 'HighChol', 'DiffWalk',
       'HeartDiseaseorAttack', 'Education_2.0', 'Education_3.0',
       'Education_4.0', 'Education_5.0', 'Education_6.0', 'diabetes_status'],
      dtype='object')

In [72]:
#Update features to match the columns incluiding the one hot encoding variable

features = [
    'BMI', 'HighBP', 'Age', 'GenHlth', 'PhysHlth', 'HighChol', 'DiffWalk',
       'HeartDiseaseorAttack', 'Education_2.0', 'Education_3.0',
       'Education_4.0', 'Education_5.0', 'Education_6.0'
]

In [73]:
# Cross-Validation

X_train_bal = train_balanced[features]  # independent variables - features
y_train_bal= train_balanced[target]  # dependent variable


In [74]:
X_train_bal.head()

,BMI,HighBP,Age,GenHlth,PhysHlth,HighChol,DiffWalk,HeartDiseaseorAttack,Education_2.0,Education_3.0,Education_4.0,Education_5.0,Education_6.0
0,27.0,1.0,13.0,1.0,0.0,1.0,0.0,0.0,False,False,False,False,True
1,32.0,1.0,10.0,5.0,15.0,1.0,1.0,0.0,False,False,True,False,False
2,25.0,1.0,13.0,2.0,30.0,0.0,0.0,1.0,False,False,False,False,True
3,27.0,1.0,9.0,5.0,30.0,1.0,0.0,0.0,False,False,False,True,False
4,24.0,0.0,4.0,1.0,0.0,0.0,0.0,0.0,False,False,False,True,False


In [75]:
X_train_bal.columns

Index(['BMI', 'HighBP', 'Age', 'GenHlth', 'PhysHlth', 'HighChol', 'DiffWalk',
       'HeartDiseaseorAttack', 'Education_2.0', 'Education_3.0',
       'Education_4.0', 'Education_5.0', 'Education_6.0'],
      dtype='object')

In [76]:
# Model 1 - logistic regression

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold, train_test_split,cross_val_score

logreg = LogisticRegression(max_iter=1000)

# Accuracy
accuracy_scores_logreg = cross_val_score(logreg, X_train_bal, y_train_bal, cv=5, scoring='accuracy')

# Precision
precision_scores_logreg = cross_val_score(logreg, X_train_bal, y_train_bal, cv=5, scoring='precision_weighted')

# Recall
recall_scores_logreg = cross_val_score(logreg, X_train_bal, y_train_bal, cv=5, scoring='recall_weighted')

# Perform cross-validation and get F1 scores
f1_scores_logreg = cross_val_score(logreg, X_train_bal, y_train_bal, cv=5, scoring='f1_weighted')


# Print metrics
print("Logistic Regression - Mean accuracy scores:", accuracy_scores_logreg.mean())
print("Logistic Regression - Mean Precision:", precision_scores_logreg.mean())
print("Logistic Regression - Mean Recall:", recall_scores_logreg.mean())
print("Logistic Regression - Mean F1 Score:", f1_scores_logreg.mean())



Logistic Regression - Mean accuracy scores: 0.737266646757943
Logistic Regression - Mean Precision: 0.7375864654212301
Logistic Regression - Mean Recall: 0.737266646757943
Logistic Regression - Mean F1 Score: 0.7371780341712526


In [77]:
# Model 2 - Decision Tree
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(max_depth = 10,  random_state=42)

# Accuracy
accuracy_scores_tree = cross_val_score(tree, X_train_bal, y_train_bal, cv=5, scoring='accuracy')

# Precision
precision_scores_tree = cross_val_score(tree, X_train_bal, y_train_bal, cv=5, scoring='precision_weighted')

# Recall
recall_scores_tree = cross_val_score(tree, X_train_bal, y_train_bal, cv=5, scoring='recall_weighted')

# Perform cross-validation and get F1 scores
f1_scores_tree = cross_val_score(tree, X_train_bal, y_train_bal, cv=5, scoring='f1_weighted')

# Print metrics
print("Decision Tree - Mean accuracy scores:", accuracy_scores_tree.mean())
print("Decision Tree - Mean Precision:", precision_scores_tree.mean())
print("Decision Tree - Mean Recall:", recall_scores_tree.mean())
print("Decision Tree - Mean F1 Score:", f1_scores_tree.mean())

Decision Tree - Mean accuracy scores: 0.7295282205549015
Decision Tree - Mean Precision: 0.729974521747032
Decision Tree - Mean Recall: 0.7295282205549015
Decision Tree - Mean F1 Score: 0.7293971394718051


In [78]:
# Model 3 - Random Forest
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators = 200, max_depth=10, random_state=42)

# Accuracy
accuracy_scores_rf = cross_val_score(rf, X_train_bal, y_train_bal, cv=5, scoring='accuracy')

# Precision
precision_scores_rf = cross_val_score(rf, X_train_bal, y_train_bal, cv=5, scoring='precision_weighted')

# Recall
recall_scores_rf = cross_val_score(rf, X_train_bal, y_train_bal, cv=5, scoring='recall_weighted')

# Perform cross-validation and get F1 scores
f1_scores_rf = cross_val_score(rf, X_train_bal, y_train_bal, cv=5, scoring='f1_weighted')

print("Random Forest - Mean accuracy scores:", accuracy_scores_rf.mean())
print("Random Forest - Mean Precision:", precision_scores_rf.mean())
print("Random Forest - Mean Recall:", recall_scores_rf.mean())
print("Random Forest - Mean F1 Score:", f1_scores_rf.mean())

Random Forest - Mean accuracy scores: 0.7386579825724169
Random Forest - Mean Precision: 0.7398548618555647
Random Forest - Mean Recall: 0.7386579825724169
Random Forest - Mean F1 Score: 0.7383319372002705


In [79]:
# Final Model 

rf.fit(X_train_bal, y_train_bal) # we train on the whole training dataset

y_pred = rf.predict(X_test)

test_accuracy = accuracy_score(y_test, y_pred)
test_precision = precision_score(y_test, y_pred)
test_recall = recall_score(y_test, y_pred)
test_f1 = f1_score(y_test, y_pred, average='weighted')  
print("Test Accuracy:", test_accuracy)
print("Test Precision:", test_precision)
print("Test Recall:", test_recall)
print("Test F1 Score:", test_f1)


Test Accuracy: 0.7174195837275308
Test Precision: 0.3316345490258534
Test Recall: 0.7814610958218664
Test F1 Score: 0.7539930361098914


**Interpretation**  
The final Random Forest model achieved an accuracy of 71.74% on the test dataset. The model demonstrated a strong recall of 78.14%, meaning that it was able to identify most individuals with diabetes/prediabetes. However, the precision was considerably lower at 33.16%, indicating that the model generated a high number of false positive predictions. In other words, although the model was successful at detecting most positive cases, many individuals predicted as having diabetes or prediabetes were not actually in that category.

Since this is a healthcare related classification problem, prioritizing recall can be beneficial because identifying individuals who may be at risk is often more important than minimizing false alarms. However, the overall performance of the model still has limitations, as the accuracy and precision values indicate that the predictions are not yet highly reliable.

Random Forest was selected as the final model because it achieved the best overall performance compared with Logistic Regression and Decision Tree models. Although the improvement over Logistic Regression was small, Random Forest consistently performed better than a single Decision Tree. Random Forest algorithm was selected because it combines multiple decision trees and uses their combined predictions to improve stability and reduce overfitting compared with an individual tree.



# Logistic Regression with L1 Regularization

In [80]:
# Logistic Regression with Lasso (L1) regularization
lasso_model = LogisticRegression(penalty='l1', solver='liblinear', C=0.01, random_state = 42)
lasso_model.fit(X_train_bal, y_train_bal)


coef_df = pd.DataFrame({
    'Feature': X_train_bal.columns,
    'Coefficient': lasso_model.coef_[0]
})

print(coef_df)
lasso_pred = lasso_model.predict(X_test)

print("Lasso (L1) Regularization Test Set Evaluation:")
print(classification_report(y_test, lasso_pred))


                 Feature  Coefficient
0                    BMI     0.065388
1                 HighBP     0.701273
2                    Age     0.141608
3                GenHlth     0.546089
4               PhysHlth    -0.005299
5               HighChol     0.554953
6               DiffWalk     0.138819
7   HeartDiseaseorAttack     0.231886
8          Education_2.0     0.000000
9          Education_3.0     0.000000
10         Education_4.0     0.000000
11         Education_5.0    -0.012034
12         Education_6.0    -0.164459
Lasso (L1) Regularization Test Set Evaluation:
              precision    recall  f1-score   support

         0.0       0.94      0.72      0.81     42742
         1.0       0.33      0.76      0.46      7994

    accuracy                           0.72     50736
   macro avg       0.64      0.74      0.64     50736
weighted avg       0.85      0.72      0.76     50736



In [81]:
# Print each metric for readability (l1)
print("Accuracy:", accuracy_score(y_test, lasso_pred))
print("Precision:", precision_score(y_test, lasso_pred))
print("Recall:", recall_score(y_test, lasso_pred))
print("F1 Score:", f1_score(y_test, lasso_pred))

Accuracy: 0.7231157363607694
Precision: 0.33353497580290364
Recall: 0.7586940205153866
F1 Score: 0.4633661853464741


The logistic regression model with L1 regularization achieved an overall acuracy of 72% on the test dataset.The model obtained a precision of 33%, a recall of 76% and an F1-score of 46%. In comparison to the final Random Forest model, Logistic regression with l1 regularization produed very similar performance. L1 regularization succesfully performed feature selection by shrinking some coefficients to exactly zero. L1 regularization shrank the coefficients of Education_2.0, Education_3.0, and Education_4.0 to zero, effectively excluding these variables from the model. Education variables may not be that informative or linked to diabetes alone, variables such as BMI,  HighChol, HighBP, are better predictors in this case.

# Logistic Regression with L2 Regularization

In [82]:
# Logistic Regression with Ridge (L2) regularization
ridge_model = LogisticRegression(penalty='l2', solver='liblinear', random_state = 42)
ridge_model.fit(X_train_bal, y_train_bal)


print(ridge_model.coef_)
coef_df = pd.DataFrame({
    'Feature': X_train_bal.columns,
    'Coefficient': ridge_model.coef_[0]
})

print(coef_df)
ridge_pred = ridge_model.predict(X_test)
print("Ridge (L2) Regularization Test Set Evaluation:")
print(classification_report(y_test, ridge_pred))


[[ 0.07259768  0.71340891  0.15309221  0.56912429 -0.00751398  0.58972884
   0.15696412  0.28241488 -0.24881698 -0.4134991  -0.52432973 -0.55042995
  -0.68537378]]
                 Feature  Coefficient
0                    BMI     0.072598
1                 HighBP     0.713409
2                    Age     0.153092
3                GenHlth     0.569124
4               PhysHlth    -0.007514
5               HighChol     0.589729
6               DiffWalk     0.156964
7   HeartDiseaseorAttack     0.282415
8          Education_2.0    -0.248817
9          Education_3.0    -0.413499
10         Education_4.0    -0.524330
11         Education_5.0    -0.550430
12         Education_6.0    -0.685374
Ridge (L2) Regularization Test Set Evaluation:
              precision    recall  f1-score   support

         0.0       0.94      0.72      0.82     42742
         1.0       0.34      0.76      0.47      7994

    accuracy                           0.73     50736
   macro avg       0.64      0.74      

In [83]:
# Print each metric for readability (l2)
print("Accuracy:", accuracy_score(y_test, ridge_pred))
print("Precision:", precision_score(y_test, ridge_pred))
print("Recall:", recall_score(y_test, ridge_pred))
print("F1 Score:", f1_score(y_test, ridge_pred))

Accuracy: 0.7264467045096185
Precision: 0.3361545743081463
Recall: 0.7551913935451589
F1 Score: 0.46522560012329983


This logistic regression model with L2 regularization achieved an accuracy of 72.60%, a precision of 33.87%, a recall of 75.28%, and a F1-score of 46.72% on the test dataset. Compared with logistic regression model with L1 regularization, the L2 model produced a slighly higher F1-score. However, the performance of these models were very small. In addition,  L2 regularization model achieved a slightly higher accuracy but lower recall. The Random Forest model correctly identified a larger proportion of individuals with diabetes, making it more suitable for a healthcare related problem where identifying individuals at risk is often more important than minimizing false positive predictions.

# Logistic Regression with Elastic Net Regularization

In [84]:
# Logistic Regression with Elastic Net regularization (Combination of L1 and L2)

elastic_net_model = LogisticRegression(penalty='elasticnet', solver='saga', l1_ratio=0.8, C=0.01, max_iter=5000, random_state=42)
elastic_net_model.fit(X_train_bal, y_train_bal)



coef_df = pd.DataFrame({
    'Feature': X_train_bal.columns,
    'Coefficient': elastic_net_model.coef_[0]
})

print(coef_df)
elastic_net_pred = elastic_net_model.predict(X_test)
print(classification_report(y_test, elastic_net_pred))


                 Feature  Coefficient
0                    BMI     0.073080
1                 HighBP     0.688310
2                    Age     0.156761
3                GenHlth     0.575750
4               PhysHlth    -0.006153
5               HighChol     0.564745
6               DiffWalk     0.118059
7   HeartDiseaseorAttack     0.227676
8          Education_2.0     0.000000
9          Education_3.0     0.000000
10         Education_4.0     0.000000
11         Education_5.0     0.000000
12         Education_6.0    -0.138177
              precision    recall  f1-score   support

         0.0       0.94      0.72      0.81     42742
         1.0       0.33      0.76      0.46      7994

    accuracy                           0.72     50736
   macro avg       0.64      0.74      0.64     50736
weighted avg       0.84      0.72      0.76     50736



In [86]:
# Print each metric for readability
print("Accuracy:", accuracy_score(y_test, elastic_net_pred))
print("Precision:", precision_score(y_test, elastic_net_pred))
print("Recall:", recall_score(y_test, elastic_net_pred))
print("F1 Score:", f1_score(y_test, elastic_net_pred))

Accuracy: 0.724909334594765
Precision: 0.3347375422648412
Recall: 0.7554415811858894
F1 Score: 0.46391396197426543


The Elastic Net model had a similar performance to the L1 and L2 Logistic regression models. All three models produced comparable accuracy, precision, recall, and F1-score. Compared with the Random Forest model, Elastic Net achieved slightly higher accuracy but slightly lower recall. In this project the primary objective is to identify individuals with diabetes, so the Random Forest model remains preferable because it identified correctly a greater proportion of positive cases.